# AI-工作台介绍

## 工作台目录结构及安装介绍
```text
ai-workbench/
├── .gitignore
├── docker-compose.yml      # PG + Redis
├── README.md
├── backend/
│   ├── .env                 # 配置（填你的 API Key）
│   ├── requirements.txt     # 锁定版本依赖
│   └── app/
│       ├── __init__.py
│       ├── config.py        # 配置加载
│       ├── database.py      # SQLAlchemy 连接
│       ├── main.py          # FastAPI 入口 + /health
│       ├── api/
│       ├── models/
│       ├── services/
│       └── utils/
└── frontend/
```

- 1. **启动 Docker 依赖**

```
# 切换到项目空间
cd D:\study\ai_study\proj\ai-workbench
# docker启动本项目同时安装相关依赖
docker-compose up -d
```
- 2. **创建 conda 环境**

```
# conda 管理环境（独立环境）
conda create -n ai-workbench python=3.11 -y
conda activate ai-workbench
```
- 3. **装依赖**（先装 torch GPU 版）

```
# 安装pytorch依赖GPU版本
pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
# 切换到后端服务
cd backend
# 安装项目所需依赖（已经有限制对应版本）
pip install -r requirements.txt
```
- 4. **改 .env** 填入你的 DashScope API Key

- 5. **启动后端**
```
uvicorn app.main:app --reload
```
- uvicorn：这是 ASGI（异步服务器网关接口）服务器。FastAPI 本身是一个 Web 框架，它不能自己监听端口处理网络请求，必须依赖 Uvicorn 这样的服务器来跑起来。
- app：项目里的文件夹（包）名叫 app。
- main：app 文件夹里面的 main.py 文件。
- app：main.py 文件中，那个用 app = FastAPI() 实例化出来的变量名。
- --reload：开发神器。开启后，只要你修改了 Python 代码并保存，Uvicorn 会自动重启服务

- 6. **前端初始化**
cd frontend
```bash
npm create vite@latest . -- --template react
npm install antd
```

- 7. **前端启动**
```bash
npm run dev
```

## 数据库表结构介绍（PostgreSQL）

### 1. documents（文档主表）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| id | UUID | 主键，自动生成 |
| title | VARCHAR(500) | 文档标题 |
| category | VARCHAR(100) | 分类（如 "RAG"、"LangGraph"） |
| tags | JSONB | 标签数组，如 `["向量检索","入门"]` |
| source | VARCHAR(50) | 来源类型：markdown /pdf/lark /jupyter |
| file_path | VARCHAR(500) | 原始文件路径 |
| md5_hash | VARCHAR(32) | 文件 MD5，增量检测用（重复上传跳过） |
| model_version | VARCHAR(20) | embedding 模型版本，如 `bge-m3-v1`，模型升级时区分新旧数据 |
| user_id | UUID | 预留，产品化时多用户用 |
| is_deleted | BOOLEAN | 软删除标记 |
| created_at | TIMESTAMP | 创建时间 |
| updated_at | TIMESTAMP | 更新时间（触发器自动） |
| deleted_at | TIMESTAMP | 删除时间 |

---

### 2. chunks（文档块表）


| 字段 | 类型 | 说明 |
| --- | --- | --- |
| id | UUID | 主键 |
| doc_id | UUID | 外键关联 documents |
| parent_chunk_id | UUID | 父块 ID，子块命中后回查父块原文 |
| chunk_index | INT | 块序号，配合 doc_id 做唯一索引 |
| page_no | INT | PDF 页码，引用定位用 |
| bbox | JSONB | PDF 位置坐标，前端高亮用 |
| chunk_type | VARCHAR(30) | 块类型：text /table/image_caption /code/output |
| source_type | VARCHAR(30) | 来源类型：markdown /jupyter_code/jupyter_output |
| content | TEXT | 块文本内容 |
| embedding | vector(1024) | bge-m3 向量，1024 维，HNSW 索引 |
| tsv | tsvector | PG 全文检索向量，GIN 索引 |
| model_version | VARCHAR(20) | embedding 模型版本 |
| is_deleted | BOOLEAN | 软删除 |
| created_at | TIMESTAMP | 创建时间 |
| updated_at | TIMESTAMP | 更新时间 |
| deleted_at | TIMESTAMP | 删除时间 |

---

### 3. skills（技能表）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| id | UUID | 主键 |
| name | VARCHAR(200) | 技能名称（如 "Dify"） |
| category | VARCHAR(100) | 分类（如 "Agent 框架"） |
| priority | INT | 优先级 1-5 |
| status | VARCHAR(30) | 状态：todo /learning/done |
| resource_url | VARCHAR(500) | 学习链接 |
| notes | TEXT | 笔记 |
| user_id | UUID | 预留 |
| is_deleted | BOOLEAN | 软删除 |
| created_at | TIMESTAMP | 创建时间 |
| updated_at | TIMESTAMP | 更新时间 |
| deleted_at | TIMESTAMP | 删除时间 |

---

### 4. daily_news（日报表）


| 字段 | 类型 | 说明 |
| --- | --- | --- |
| id | UUID | 主键 |
| news_date | DATE | 日期，唯一 |
| summary | TEXT | AI 总结的日报 |
| articles | JSONB | 原始文章列表 |
| created_at | TIMESTAMP | 创建时间 |
| updated_at | TIMESTAMP | 更新时间 |

---

### 5. learning_paths（学习路径表）


| 字段 | 类型 | 说明 |
| --- | --- | --- |
| id | UUID | 主键 |
| goal | VARCHAR(200) | 目标岗位 |
| nodes | JSONB | 学习路径树 |
| created_at | TIMESTAMP | 创建时间 |
| updated_at | TIMESTAMP | 更新时间 |

---

### 核心关系

```
documents 1 ── N chunks
                └── parent_chunk_id 自关联（父子块）
```

In [ ]:
%%sql
